In [7]:
from tqdm import tqdm_notebook
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nb
from sklearn.svm import SVC
from nilearn import image
from scipy.interpolate import interp1d

mask_names = ['mofc','lofc','hp','dlPFC','FFA/PPA']

subject_inds = [1,2,3,4,5,7,8,9,10,11,12,14,15,16,17,18,20,21,22,23,24,25,26,27,28,29,30]
index = pd.MultiIndex.from_product([subject_inds, mask_names], names=['subject', 'mask_name'])
df_scores = pd.DataFrame(index=index, columns=['score', 'chance_score'])
categories = np.array(['(Fy)Fo', '(Fo)Fy', '(Hy)Ho', '(Ho)Hy', '(Fy)Fy', '(Fo)Fo', '(Hy)Hy', '(Ho)Ho', '(Fy)Hy', '(Fy)Ho', '(Fo)Hy', '(Fo)Ho', '(Hy)Fy', '(Hy)Fo', '(Ho)Fy', '(Ho)Fo'])



for current_subject in tqdm_notebook(subject_inds):
    print("The current subject is %s" % current_subject)
    data_dir = f'/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/'
    all_beta = f"{data_dir}/sub-{current_subject:02}_state.nii.gz"
    datasets = {"mofc":[f"/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/mofc_bin.nii.gz"],
                "lofc":[f"/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/lofc_bin.nii.gz"],
                "hp":[f"/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/hp_bin.nii.gz"],
                "dlPFC":[f"/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/dlPfC_bin.nii.gz"],
                "FFA/PPA":[f"/home/garlicseed/Mount/final/beta/sub-{current_subject:02}/FFA-PPA_bin.nii.gz"],                
               }
    labels = pd.read_csv('/home/garlicseed/Mount/final/beta/state_events.tsv',sep='\t')
    
    labels_run = labels['run']
    #categories = labels['state'].unique()
    session_labels = labels_run
    state_beta = nb.load(all_beta) 
    state = labels['state']
    
    from sklearn.model_selection import LeaveOneGroupOut
    logo = LeaveOneGroupOut()
    from sklearn.preprocessing import LabelEncoder
    encoder = LabelEncoder()
    state = encoder.fit_transform(state)
    from nilearn.maskers import NiftiMasker
    
    for mask_name in tqdm_notebook(mask_names):
        print("Working on %s " % mask_name)
        mask_filename = datasets[mask_name][0]
        masker = NiftiMasker(
            mask_img=mask_filename,
            runs=session_labels,
            standardize="zscore_sample",
            smoothing_fwhm=4,
            memory="nilearn_cache",
            memory_level=1,
                        )
        fmri_masked = masker.fit_transform(state_beta)

        from sklearn.feature_selection import SelectKBest, f_classif
        from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
        from sklearn.preprocessing import StandardScaler 
        from sklearn.model_selection import GridSearchCV
        from sklearn.model_selection import cross_val_score
        from sklearn.pipeline import Pipeline
        from sklearn.svm import SVC
        from sklearn.dummy import DummyClassifier
        from sklearn.metrics import accuracy_score
        
        svc_ova = OneVsRestClassifier(
            Pipeline(
                [
                    ("anova", SelectKBest(f_classif, k='all')),
                    #('scaler', StandardScaler()),
                    ("svc", SVC(kernel='linear',C=0.001)),
                ]
                 )
              )
        performance_svc = []
        performance_dummy = []
        for train_idx, test_idx in logo.split(fmri_masked, state, session_labels):
            X_train, X_test = fmri_masked[train_idx, :], fmri_masked[test_idx, :]
            y_train, y_test = state[train_idx], state[test_idx]
            svc_ova.fit(X_train, y_train)
            dummy = DummyClassifier()
            dummy.fit(X_train, y_train)
            y_pred = svc_ova.predict(X_test)
            y_pred = np.squeeze(y_pred)
            performance_svc.append(accuracy_score(y_test, y_pred))
            
            y_pred_dummy = dummy.predict(X_test)
            y_pred_dummy = np.squeeze(y_pred_dummy)
            performance_dummy.append(accuracy_score(y_test, y_pred_dummy)) 
        y_pred_final_svc = np.mean(performance_svc, axis=0)
        y_pred_final_dummy = np.mean(performance_dummy, axis=0)
        df_scores.loc[(current_subject, mask_name), 'score'] = y_pred_final_svc
        df_scores.loc[(current_subject, mask_name), 'chance_score'] = y_pred_final_dummy
        
        print("Scores: %1.2f +- %1.2f" % (np.mean(performance_svc), np.std(performance_svc)))
        print("Chacne Scores: %1.2f +- %1.2f" % (np.mean(performance_dummy), np.std(performance_dummy)))

df_scores = df_scores.reset_index()  
df_mean = df_scores.groupby(['mask_name'])[['score', 'chance_score']].mean()
df_mean = df_mean.reset_index()
df_scores.to_csv('/home/garlicseed/Mount/final/MVPA/16way_classifier/Scores/accuray.tsv',sep='\t')


  0%|          | 0/27 [00:00<?, ?it/s]

The current subject is 1


  0%|          | 0/5 [00:00<?, ?it/s]

Working on mofc 
Scores: 0.09 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on lofc 
Scores: 0.06 +- 0.04
Chacne Scores: 0.06 +- 0.00
Working on hp 
Scores: 0.11 +- 0.11
Chacne Scores: 0.06 +- 0.00
Working on dlPFC 
Scores: 0.11 +- 0.05
Chacne Scores: 0.06 +- 0.00
Working on FFA/PPA 
Scores: 0.08 +- 0.05
Chacne Scores: 0.06 +- 0.00
The current subject is 2


  0%|          | 0/5 [00:00<?, ?it/s]

...

Working on mofc 
Scores: 0.02 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on lofc 
Scores: 0.09 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on hp 
Scores: 0.09 +- 0.10
Chacne Scores: 0.06 +- 0.00
Working on dlPFC 
Scores: 0.12 +- 0.04
Chacne Scores: 0.06 +- 0.00
Working on FFA/PPA 
Scores: 0.12 +- 0.04
Chacne Scores: 0.06 +- 0.00
The current subject is 29


  0%|          | 0/5 [00:00<?, ?it/s]

Working on mofc 
Scores: 0.16 +- 0.11
Chacne Scores: 0.06 +- 0.00
Working on lofc 
Scores: 0.17 +- 0.08
Chacne Scores: 0.06 +- 0.00
Working on hp 
Scores: 0.16 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on dlPFC 
Scores: 0.34 +- 0.05
Chacne Scores: 0.06 +- 0.00
Working on FFA/PPA 
Scores: 0.17 +- 0.14
Chacne Scores: 0.06 +- 0.00
The current subject is 30


  0%|          | 0/5 [00:00<?, ?it/s]

Working on mofc 
Scores: 0.03 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on lofc 
Scores: 0.03 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on hp 
Scores: 0.11 +- 0.03
Chacne Scores: 0.06 +- 0.00
Working on dlPFC 
Scores: 0.12 +- 0.10
Chacne Scores: 0.06 +- 0.00
Working on FFA/PPA 
Scores: 0.12 +- 0.04
Chacne Scores: 0.06 +- 0.00
